# Test de positionnement — notebook de vérification

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/positionnement/02_notebook.ipynb)

Fais d'abord le test **sur papier** (`01_sujet.md`, environ 2 heures, sans fiche ni calculatrice). Reporte ensuite ici **les valeurs** que tu as trouvées, exercice par exercice, sous la forme rappelée au-dessus de chaque cellule. Une question que tu n'as pas su faire : laisse `...`.

Chaque cellule vérifie tes réponses avec `wbm.check` et affiche ✅ ou ❌, sans le message d'erreur tant que `FINISHED` vaut `False` : le test mesure ce que tu savais sur papier. Seul un problème de **forme** est expliqué tout de suite (⚠️ : guillemets, virgule, fraction tapée en décimal) : corrige l'écriture, jamais la valeur. Une erreur rouge (`SyntaxError`, `NameError`) vient aussi de la forme : il manque sans doute des guillemets.

Quand tout est reporté, mets `FINISHED = True` dans la cellule des outils, puis « Tout exécuter » : les messages s'affichent, et la dernière cellule donne ta grille, chapitre par chapitre.

> Travaille dans **ta copie** (`mon_travail/positionnement/02_notebook.ipynb`, créée par `python tools/start_chapter.py POS`) ; la version du dépôt, `positionnement/02_notebook.ipynb`, est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="POS")  # your own library

## Les outils

Exécute cette cellule (« Tout exécuter » le fait pour toi). Mets `FINISHED = True` quand tu as reporté toutes tes réponses.

In [ ]:
# Tools of the placement test: they store each check, then turn the results into your grid.
from fractions import Fraction

FINISHED = False   # put True when you have reported ALL your answers (a question left at ... then counts as missed)
RESULTS = {}       # filled by the check cells below: exercise -> [(right?, status)] for its questions

GRID = {"R1": ("Nombres et calcul", ["POS.1", "POS.2"]),
        "R2": ("Calcul littéral et équations", ["POS.3", "POS.4"]),
        "R3": ("Fonctions", ["POS.5", "POS.6"]),
        "R4": ("Logique, ensembles, Σ et Π", ["POS.7", "POS.8"]),
        "R5": ("Suites", ["POS.9", "POS.10"]),
        "R6": ("Exponentielle et logarithmes", ["POS.11", "POS.12"]),
        "R7": ("Limites et dérivation", ["POS.13", "POS.14"]),
        "R8": ("Probabilités de base", ["POS.15", "POS.16"]),
        "R9": ("Géométrie du plan, vecteurs et trigonométrie", ["POS.17", "POS.18"])}
ADVICE = {2: "tu peux sauter", 1: "à survoler", 0: "à faire en entier"}


def is_form_problem(result):
    """A message about the way the answer is written (quotes, comma, fraction typed as a decimal), not its value."""
    return result.status == "type" or (result.status == "near" and "valeur approchée" in result.message)


def check_all(ex_id, letters, answers):
    """Check every question of a placement exercise and return (right?, status) for each.

    Until FINISHED is True, a wrong value only shows ❌: its message could guide a correction, and the test
    measures what you knew on paper. A problem of form (quotes, decimal point...) is always explained.
    """
    results = []
    for letter, answer in zip(letters, answers):
        result = wbm.check(f"{ex_id}{letter}", answer, quiet=True)
        if result.status == "pending":
            line = "— laissée vide : comptée fausse" if FINISHED else "⏳ vide (laisse `...` si tu n'as pas su la faire)"
        elif result.passed:
            line = "✅"
        elif is_form_problem(result):
            line = f"⚠️ la forme, pas la valeur : {result.message} Corrige l'écriture, garde ta valeur."
        elif FINISHED:
            line = f"❌ {result.message}"
        else:
            line = "❌ (le message s'affichera quand FINISHED = True)"
        print(f"{ex_id}{letter} {line}")
        results.append((bool(result.passed), result.status))
    return results


def show_grid(results, finished):
    """Print the verdict of each chapter: an exercise counts when ALL its questions are right."""
    for cid, (title, exercises) in GRID.items():
        checks = [results.get(ex, [(False, "pending")]) for ex in exercises]
        if not finished and any(status == "pending" for c in checks for _, status in c):
            verdict = "⏳ pas encore rempli (ou mets FINISHED = True dans la cellule des outils)"
        else:
            score = sum(all(right for right, _ in c) for c in checks)
            verdict = f"{score}/2 → {ADVICE[score]}"
        print(f"{cid} · {title:<46} {verdict}")
    print("\nCe que veut dire chaque verdict, et ce qu'il faut faire quand même : positionnement/05_grille.md")

## R1 · Nombres et calcul

**POS.1 — Calculer avec des priorités et des fractions** · a) un nombre · b) une fraction irréductible, entre guillemets · c) une fraction irréductible, entre guillemets · d) une fraction irréductible, entre guillemets

In [ ]:
answer_POS_1a = ...  # a) a number
answer_POS_1b = ...  # b) an irreducible fraction, as text
answer_POS_1c = ...  # c) an irreducible fraction, as text
answer_POS_1d = ...  # d) an irreducible fraction, as text

RESULTS["POS.1"] = check_all("POS.1", "abcd", [answer_POS_1a, answer_POS_1b, answer_POS_1c, answer_POS_1d])

**POS.2 — Puissances, racines et notation scientifique** · a) un nombre · b) un entier · c) un nombre · d) un décimal, avec un point

In [ ]:
answer_POS_2a = ...  # a) a number
answer_POS_2b = ...  # b) an integer
answer_POS_2c = ...  # c) a number
answer_POS_2d = ...  # d) a decimal

RESULTS["POS.2"] = check_all("POS.2", "abcd", [answer_POS_2a, answer_POS_2b, answer_POS_2c, answer_POS_2d])

## R2 · Calcul littéral et équations

**POS.3 — Développer, factoriser, résoudre une équation et une inéquation** · a) la liste `[a, b, c]` · b) un nombre · c) un nombre · d) un intervalle, entre guillemets

In [ ]:
answer_POS_3a = ...  # a) the list [a, b, c]
answer_POS_3b = ...  # b) a number
answer_POS_3c = ...  # c) a number
answer_POS_3d = ...  # d) an interval, as text

RESULTS["POS.3"] = check_all("POS.3", "abcd", [answer_POS_3a, answer_POS_3b, answer_POS_3c, answer_POS_3d])

**POS.4 — Un système de deux équations à deux inconnues** · a) la liste `[x, y]` · b) le numéro 1, 2 ou 3

In [ ]:
answer_POS_4a = ...  # a) the list [x, y]
answer_POS_4b = ...  # b) 1, 2 or 3

RESULTS["POS.4"] = check_all("POS.4", "ab", [answer_POS_4a, answer_POS_4b])

## R3 · Fonctions

**POS.5 — Lire un graphique de fonction** · a) un nombre · b) un nombre · c) un intervalle, entre guillemets · d) un nombre

In [ ]:
answer_POS_5a = ...  # a) a number
answer_POS_5b = ...  # b) a number
answer_POS_5c = ...  # c) an interval, as text
answer_POS_5d = ...  # d) a number

RESULTS["POS.5"] = check_all("POS.5", "abcd", [answer_POS_5a, answer_POS_5b, answer_POS_5c, answer_POS_5d])

**POS.6 — Racines et sommet d'un polynôme du second degré** · a) un nombre · b) une liste de nombres · c) une fraction (entre guillemets) ou un décimal · d) une fraction (entre guillemets) ou un décimal

In [ ]:
answer_POS_6a = ...  # a) a number
answer_POS_6b = ...  # b) a list of numbers
answer_POS_6c = ...  # c) a fraction or a decimal
answer_POS_6d = ...  # d) a fraction or a decimal

RESULTS["POS.6"] = check_all("POS.6", "abcd", [answer_POS_6a, answer_POS_6b, answer_POS_6c, answer_POS_6d])

## R4 · Logique, ensembles, Σ et Π

**POS.7 — Sommes avec Σ, somme double et produit avec Π** · a) un nombre · b) un nombre · c) un nombre · d) un nombre

In [ ]:
answer_POS_7a = ...  # a) a number
answer_POS_7b = ...  # b) a number
answer_POS_7c = ...  # c) a number
answer_POS_7d = ...  # d) a number

RESULTS["POS.7"] = check_all("POS.7", "abcd", [answer_POS_7a, answer_POS_7b, answer_POS_7c, answer_POS_7d])

**POS.8 — La négation d'une phrase avec « pour tout »** · a) le numéro 1, 2, 3 ou 4 · b) `True` ou `False`

In [ ]:
answer_POS_8a = ...  # a) 1, 2, 3 or 4
answer_POS_8b = ...  # b) True or False

RESULTS["POS.8"] = check_all("POS.8", "ab", [answer_POS_8a, answer_POS_8b])

## R5 · Suites

**POS.9 — Terme et somme d'une suite géométrique** · a) un nombre · b) un nombre · c) une fraction, entre guillemets

In [ ]:
answer_POS_9a = ...  # a) a number
answer_POS_9b = ...  # b) a number
answer_POS_9c = ...  # c) a fraction, as text

RESULTS["POS.9"] = check_all("POS.9", "abc", [answer_POS_9a, answer_POS_9b, answer_POS_9c])

**POS.10 — Une suite définie par récurrence : premiers termes et limite** · a) la liste `[v1, v2, v3]` · b) un nombre

In [ ]:
answer_POS_10a = ...  # a) the list [v1, v2, v3]
answer_POS_10b = ...  # b) a number

RESULTS["POS.10"] = check_all("POS.10", "ab", [answer_POS_10a, answer_POS_10b])

## R6 · Exponentielle et logarithmes

**POS.11 — Propriétés de l'exponentielle et du logarithme** · a) un nombre · b) un entier · c) une fraction (entre guillemets) ou un décimal · d) un nombre

In [ ]:
answer_POS_11a = ...  # a) a number
answer_POS_11b = ...  # b) an integer
answer_POS_11c = ...  # c) a fraction or a decimal
answer_POS_11d = ...  # d) a number

RESULTS["POS.11"] = check_all("POS.11", "abcd", [answer_POS_11a, answer_POS_11b, answer_POS_11c, answer_POS_11d])

**POS.12 — Résoudre des équations avec exp et ln** · a) une expression exacte avec `ln`, entre guillemets · b) une expression exacte avec `exp`, entre guillemets · c) un nombre

In [ ]:
answer_POS_12a = ...  # a) an exact expression with ln, as text
answer_POS_12b = ...  # b) an exact expression with exp, as text
answer_POS_12c = ...  # c) a number

RESULTS["POS.12"] = check_all("POS.12", "abc", [answer_POS_12a, answer_POS_12b, answer_POS_12c])

## R7 · Limites et dérivation

**POS.13 — Dériver trois fonctions, dont une composée** · a) une expression en x, entre guillemets · b) une expression en x, entre guillemets · c) une expression en x, entre guillemets

In [ ]:
answer_POS_13a = ...  # a) an expression in x, as text
answer_POS_13b = ...  # b) an expression in x, as text
answer_POS_13c = ...  # c) an expression in x, as text

RESULTS["POS.13"] = check_all("POS.13", "abc", [answer_POS_13a, answer_POS_13b, answer_POS_13c])

**POS.14 — Tangente, variations et une limite** · a) la liste `[m, p]` · b) un intervalle, entre guillemets · c) un nombre · d) un nombre

In [ ]:
answer_POS_14a = ...  # a) the list [m, p]
answer_POS_14b = ...  # b) an interval, as text
answer_POS_14c = ...  # c) a number
answer_POS_14d = ...  # d) a number

RESULTS["POS.14"] = check_all("POS.14", "abcd", [answer_POS_14a, answer_POS_14b, answer_POS_14c, answer_POS_14d])

## R8 · Probabilités de base

**POS.15 — Une probabilité avec un arbre** · a) une fraction irréductible, entre guillemets · b) une fraction irréductible, entre guillemets

In [ ]:
answer_POS_15a = ...  # a) an irreducible fraction, as text
answer_POS_15b = ...  # b) an irreducible fraction, as text

RESULTS["POS.15"] = check_all("POS.15", "ab", [answer_POS_15a, answer_POS_15b])

**POS.16 — L'espérance et la variance d'une variable aléatoire** · a) un décimal, avec un point · b) un décimal, avec un point

In [ ]:
answer_POS_16a = ...  # a) a decimal
answer_POS_16b = ...  # b) a decimal

RESULTS["POS.16"] = check_all("POS.16", "ab", [answer_POS_16a, answer_POS_16b])

## R9 · Géométrie du plan, vecteurs et trigonométrie

**POS.17 — Distance et produit scalaire dans le plan** · a) un nombre · b) un nombre · c) `True` ou `False`

In [ ]:
answer_POS_17a = ...  # a) a number
answer_POS_17b = ...  # b) a number
answer_POS_17c = ...  # c) True or False

RESULTS["POS.17"] = check_all("POS.17", "abc", [answer_POS_17a, answer_POS_17b, answer_POS_17c])

**POS.18 — Cosinus d'un angle et radians** · a) une fraction, entre guillemets · b) une fraction, entre guillemets · c) un nombre (en degrés, sans le signe °)

In [ ]:
answer_POS_18a = ...  # a) a fraction, as text
answer_POS_18b = ...  # b) a fraction, as text
answer_POS_18c = ...  # c) a number in degrees

RESULTS["POS.18"] = check_all("POS.18", "abc", [answer_POS_18a, answer_POS_18b, answer_POS_18c])

## Ta grille

Un exercice compte quand **toutes** ses questions sont justes. Deux exercices par chapitre : 2/2 → tu peux sauter, 1/2 → à survoler, 0/2 → à faire en entier. Le détail, et ce qu'il faut faire quand même, est dans `05_grille.md`.

In [ ]:
show_grid(RESULTS, FINISHED)

## Et maintenant

Recopie ta grille dans ta copie de `04_mes_reponses.md` et dans `mon_travail/suivi/journal.md`. Lis `03_corrige.md` pour les questions ratées : chacune renvoie à la section du chapitre qui l'enseigne. Puis passe à R1 : même « sauté », un chapitre garde quelques exercices à faire (`05_grille.md`).